# 1. Find, Select Data, and Data Processing (load, explore datasetâ€¦)

In [ ]:
# Dataset files are loaded from the local data directory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
movies = pd.read_csv('data/movie.csv')
ratings = pd.read_csv('data/rating.csv')

In [ ]:
movies.head()

In [ ]:
ratings.head()

In [ ]:
data = pd.pivot(index = 'movieId',columns = 'userId', data = ratings,values ='rating')
data.head()

In [ ]:
numberOf_user_voted_for_movie = pd.DataFrame(ratings.groupby('movieId')['rating'].agg('count'))
numberOf_user_voted_for_movie.reset_index(level = 0,inplace = True)
numberOf_user_voted_for_movie.head()

In [ ]:
data.shape

In [ ]:
numberOf_movies_voted_by_user = pd.DataFrame(ratings.groupby('userId')['rating'].agg('count'))
numberOf_movies_voted_by_user.reset_index(level = 0,inplace = True)
numberOf_movies_voted_by_user.head()

In [ ]:
data.fillna(0,inplace = True)
data.head()

In [ ]:
numberOf_user_voted_for_movie.describe()

In [ ]:
numberOf_movies_voted_by_user.describe()

In [ ]:
plt.figure()
ax = sns.scatterplot(y = 'rating', x = 'movieId', data = numberOf_user_voted_for_movie)
plt.axhline(y=10,color='r')
plt.ylabel('Number Of Users Voted for Movie')

In [ ]:
plt.figure()
ax = sns.scatterplot(y = 'rating', x = 'userId', data = numberOf_movies_voted_by_user)
plt.axhline(y=60,color='r')
plt.ylabel('Number Of Movies rated by user')

In [ ]:
data_final = data.loc[numberOf_user_voted_for_movie[numberOf_user_voted_for_movie['rating'] > 10]['movieId'],:]
data_final = data_final.loc[:,numberOf_movies_voted_by_user[numberOf_movies_voted_by_user['rating'] > 60]['userId']]
data_final.shape

In [ ]:
data_final

In [ ]:
from scipy.sparse import csr_matrix
csr_data = csr_matrix(data_final.values)
data_final.reset_index(inplace=True)

In [ ]:
data_final.head()

# 3. Implementation: UsingÂ numpy,Â pandas, andÂ sklearn

In [ ]:
from sklearn.neighbors import NearestNeighbors
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20)
knn.fit(csr_data)
def get_movie_recommendation(movie_name):
    n= 10
    movie_list = movies[movies['title'].str.contains(movie_name)]
    if len(movie_list):
        movie_idx= movie_list.iloc[0]['movieId'] #movieId
        movie_idx = data_final[data_final['movieId'] == movie_idx].index[0] #userId acc to movieId
        distances , indices = knn.kneighbors(csr_data[movie_idx],n_neighbors=n+1)
        rec_movie_indices = sorted(list(zip(indices.squeeze(),distances.squeeze())),key=lambda x: x[1])[1::1]
        recommend = []
        recommend2 = []
        for val in rec_movie_indices:
            movie_idx = data_final.iloc[val[0]]['movieId']
            idx = movies[movies['movieId'] == movie_idx].index
            recommend.append(movies.iloc[idx]['title'].values[0])
            recommend2.append(val[1])
        df1 = pd.DataFrame(recommend)
        df2 = pd.DataFrame(recommend2)
        df = pd.concat([df1,df2],axis = 'columns')
        df.columns = ['Title','Distance']
        df.set_index('Distance',inplace = True)
        return df
    else:
        return "No movies found. Please check your input"

In [ ]:
# n = input()
# get_movie_recommendation(n.title())
get_movie_recommendation('Toy Story')

# Front-end deployment

In [ ]:
# Install necessary dependencies with forced reinstallation
!pip install --ignore-installed itsdangerous blinker
!pip install --no-deps flask

In [ ]:
# Dataset files are loaded from the local data directory.

# 2. Implementation: Using the Surprise library

In [ ]:
!pip install scikit-surprise

In [ ]:
# Data Sampling and Preparation (Do this once)
ratings_sampled = ratings.sample(n=100000, random_state=42)

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_sampled[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
from surprise import KNNBasic, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define KNN parameters
sim_options = {
    'name': 'cosine',
    'user_based': False,
}

knn_model = KNNBasic(sim_options=sim_options, min_k=1, k=10)

# Training and testing
try:
    print("Starting model training...")
    knn_model.fit(trainset)
    print("Training successful.")

    predictions = knn_model.test(testset)

    rmse = accuracy.rmse(predictions)
    print(f"RMSE on test set: {rmse:.4f}")

except ZeroDivisionError as e:
    print("A ZeroDivisionError occurred during computation.")

print("Testing completed.")


In [ ]:
from surprise import KNNBasic, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define parameters for the KNNBasic model with a new similarity measure
sim_options = {
    'name': 'pearson_baseline',
    'user_based': False,
}

# Create KNNBasic model with parameters
knn_model = KNNBasic(sim_options=sim_options, min_k=1, k=10)

# Add print statements to check each step
try:
    print("Starting model training...")
    knn_model.fit(trainset)
    print("Training successful.")

    # Predict on the test set
    predictions = knn_model.test(testset)

    # Calculate and print RMSE
    rmse = accuracy.rmse(predictions)
    print(f"RMSE on test set: {rmse:.4f}")

except ZeroDivisionError as e:
    print("A ZeroDivisionError occurred during the computation of the similarity matrix.")

    # Access the user-item matrix to see if any items have all zero ratings
    user_item_matrix = ratings_sampled.pivot(index='userId', columns='movieId', values='rating').fillna(0)

    zero_columns = user_item_matrix.columns[(user_item_matrix == 0).all()]

    if len(zero_columns) > 0:
        print(f"The following items have no ratings (all values are 0): {zero_columns.tolist()}")
    else:
        print("There are no items with all values as 0.")

print("Testing completed.")


In [ ]:
from surprise import AlgoBase, Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
import pandas as pd
import numpy as np
from scipy.spatial.distance import cityblock, euclidean

# Load 'movies' and 'ratings' DataFrames
# Assuming they are already loaded

# Sample the data to reduce computation time
ratings_sampled = ratings.sample(n=100000, random_state=42)

# Prepare the data for Surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_sampled[['userId', 'movieId', 'rating']], reader)

# Split the data into training and test sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Get the set of users and items in the training set
train_users = set([trainset.to_raw_uid(u) for u in range(trainset.n_users)])
train_items = set([trainset.to_raw_iid(i) for i in range(trainset.n_items)])

# Filter the test set to include only users and items present in the training set
filtered_testset = [(uid, iid, r) for (uid, iid, r) in testset if uid in train_users and iid in train_items]

# Define L1 and L2 distance functions
def l1_distance(x, y):
    return np.sum(np.abs(x - y))  # Vectorized Manhattan distance (L1)

def l2_distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2))  # Vectorized Euclidean distance (L2)

# Create a custom KNN class using L1 or L2 distances
class KNNCustom(AlgoBase):
    def __init__(self, distance_func, k=10):
        super().__init__()
        self.distance_func = distance_func
        self.k = k

    def fit(self, trainset):
        super().fit(trainset)
        self.trainset = trainset

        # Build the item-user matrix for item-based filtering
        self.item_user_matrix = np.zeros((trainset.n_items, trainset.n_users))
        for uid, iid, rating in trainset.all_ratings():
            self.item_user_matrix[int(iid), int(uid)] = rating
        return self

    def estimate(self, u, i):
        if not self.trainset.knows_user(u):
            return self.trainset.global_mean
        if not self.trainset.knows_item(i):
            return self.trainset.global_mean

        # Get the items rated by the user
        user_ratings = self.trainset.ur[u]  # List of (item_inner_id, rating)
        user_items = [item_id for (item_id, _) in user_ratings]

        item_vector = self.item_user_matrix[i]
        distances = []

        # Compute distance to items the user has rated
        for neighbor_item in user_items:
            if neighbor_item != i:
                neighbor_vector = self.item_user_matrix[neighbor_item]
                dist = self.distance_func(item_vector, neighbor_vector)
                distances.append((dist, neighbor_item))

        # Sort and select k nearest neighbors
        distances.sort(key=lambda x: x[0])
        k_nearest_neighbors = distances[:self.k]

        # Convert user_ratings to a dictionary for fast lookup
        user_ratings_dict = dict(user_ratings)

        # Compute prediction using ratings from similar items
        sum_ratings, sum_weights = 0, 0
        for dist, neighbor_item in k_nearest_neighbors:
            neighbor_rating = user_ratings_dict.get(neighbor_item)
            if neighbor_rating is not None:
                weight = 1.0 if dist == 0 else 1.0 / dist
                sum_ratings += neighbor_rating * weight
                sum_weights += weight

        if sum_weights == 0:
            # Return the user's mean rating if no similar items are found
            return np.mean([r for (_, r) in user_ratings])
        else:
            return sum_ratings / sum_weights

# Compare methods (Cosine, Pearson, Pearson Baseline, MSD, L1, L2)
similarity_options = [
    {'name': 'cosine', 'user_based': False},
    {'name': 'pearson', 'user_based': False},
    {'name': 'pearson_baseline', 'user_based': False},
    {'name': 'msd', 'user_based': False},
]

# Initialize a dictionary to store RMSE results for each method
rmse_results = {}

for sim_option in similarity_options:
    sim_name = sim_option['name']
    print(f"\nTraining KNN model with {sim_name} similarity...")

    # Create and train KNNBasic model with each similarity measure
    knn_model = KNNBasic(sim_options=sim_option, k=10, min_k=1)
    knn_model.fit(trainset)

    # Predict on the filtered test set and compute RMSE
    predictions = knn_model.test(filtered_testset)
    rmse = accuracy.rmse(predictions, verbose=True)
    rmse_results[sim_name] = rmse

# Test with L1 and L2 distances
custom_distance_methods = {'L1': l1_distance, 'L2': l2_distance}

for dist_name, dist_func in custom_distance_methods.items():
    print(f"\nTraining KNN model with {dist_name} distance...")
    knn_custom_model = KNNCustom(distance_func=dist_func, k=10)
    knn_custom_model.fit(trainset)

    # Predict on the filtered test set and compute RMSE
    predictions = knn_custom_model.test(filtered_testset)
    rmse = accuracy.rmse(predictions, verbose=True)
    rmse_results[dist_name] = rmse

# Compare RMSE results of each method
print("\nComparison of RMSE results for different similarity measures:")
for sim_name, rmse_value in rmse_results.items():
    print(f"{sim_name} similarity/distance: RMSE = {rmse_value:.4f}")


In [ ]:
from surprise import KNNBasic, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define similarity options for KNNBasic
sim_option = {
    'name': 'cosine',     # Use cosine similarity
    'user_based': True,   # True for user-based collaborative filtering
}

# Create the KNNBasic model with the parameters
knn_model = KNNBasic(sim_options=sim_option, min_k=1, k=10)

# Train the model
print("Starting model training...")
knn_model.fit(trainset)
print("Training successful.")

# Function to predict rating for a specific userId and movieId
def predict_rating_for_user_item(user_id, movie_id):
    try:
        prediction = knn_model.predict(user_id, movie_id)
        print(f"Prediction for user_id={user_id}, movie_id={movie_id}: {prediction.est:.4f}")
    except Exception as e:
        print(f"Error during prediction: {e}")

# Select a userId and movieId that exist in the training set
# Get all userIds and movieIds from the training set
train_user_ids = set([trainset.to_raw_uid(i) for i in range(trainset.n_users)])
train_movie_ids = set([trainset.to_raw_iid(i) for i in range(trainset.n_items)])

# Pick a random userId and movieId from the training set
user_id_input = np.random.choice(list(train_user_ids))
movie_id_input = np.random.choice(list(train_movie_ids))

# Ensure IDs are in the correct type (int)
user_id_input = int(user_id_input)
movie_id_input = int(movie_id_input)

# Predict the rating
print(f"Predicting rating for user_id={user_id_input}, movie_id={movie_id_input}")
predict_rating_for_user_item(user_id_input, movie_id_input)

# Evaluate on the test set and compute RMSE
print("\nEvaluating model on the test set...")
predictions = knn_model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"RMSE on test set: {rmse:.4f}")


# **3. Implementation: Use numpy, pandas, and sklearn.**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Optional: Sample the data to reduce computation time
ratings_sampled = ratings.sample(n=100000, random_state=42)

# Split the data into training and test sets
train_data, test_data = train_test_split(ratings_sampled, test_size=0.2, random_state=42)

# Create a user-item interaction matrix for the training data
train_user_item_matrix = train_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Create a user-item interaction matrix for the test data (for evaluation purposes)
test_user_item_matrix = test_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Ensure that the userIds and movieIds in the test set are present in the training set
test_data = test_data[test_data['userId'].isin(train_user_item_matrix.index) &
                      test_data['movieId'].isin(train_user_item_matrix.columns)]

# Re-create the test_user_item_matrix after filtering
test_user_item_matrix = test_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Compute cosine similarity between users
def cosine_similarity(matrix):
    # Normalize the matrix (to handle users with different rating scales)
    norm_matrix = matrix - matrix.mean(axis=1).values.reshape(-1, 1)
    # Compute the dot product between users
    sim_matrix = np.dot(norm_matrix, norm_matrix.T)
    # Compute the magnitude of vectors for each user
    magnitude = np.sqrt(np.sum(norm_matrix ** 2, axis=1))
    # Compute the outer product of magnitudes
    mag_product = np.outer(magnitude, magnitude)
    # Handle division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        sim_matrix = np.divide(sim_matrix, mag_product)
        sim_matrix[np.isnan(sim_matrix)] = 0  # Replace NaN with 0
    # Set diagonal to 0 to exclude self-similarity
    np.fill_diagonal(sim_matrix, 0)
    return pd.DataFrame(sim_matrix, index=matrix.index, columns=matrix.index)

# Create the user similarity matrix
user_similarity = cosine_similarity(train_user_item_matrix)

# Function to find k nearest neighbors for a user
def find_knn(user_id, k):
    if user_id in user_similarity.index:
        sim_scores = user_similarity.loc[user_id]
        knn_users = sim_scores.nlargest(k).index.values
        return knn_users
    else:
        return []

# Predict rating for a user and item using KNN collaborative filtering
def predict_rating(user_id, item_id, k=5):
    if user_id not in train_user_item_matrix.index or item_id not in train_user_item_matrix.columns:
        # Return global mean if user or item is unknown
        return train_data['rating'].mean()

    knn_users = find_knn(user_id, k)

    # Get the ratings of the k nearest neighbors for the item
    neighbor_ratings = train_user_item_matrix.loc[knn_users, item_id]

    # Get the similarity scores
    sim_scores = user_similarity.loc[user_id, knn_users]

    # Filter out neighbors who haven't rated the item
    mask = neighbor_ratings > 0
    neighbor_ratings = neighbor_ratings[mask]
    sim_scores = sim_scores[mask]

    if sim_scores.sum() > 0:
        predicted_rating = np.dot(sim_scores, neighbor_ratings) / sim_scores.sum()
    else:
        # If no neighbors have rated the item, return the user's average rating
        predicted_rating = train_user_item_matrix.loc[user_id].replace(0, np.NaN).mean()
        if np.isnan(predicted_rating):
            # If the user hasn't rated any items, return the global mean
            predicted_rating = train_data['rating'].mean()

    return predicted_rating

# Evaluate the model on the test set
y_true = []
y_pred = []

print("Evaluating the model...")

for index, row in test_data.iterrows():
    user_id = row['userId']
    item_id = row['movieId']
    true_rating = row['rating']
    predicted_rating = predict_rating(user_id, item_id, k=5)
    y_true.append(true_rating)
    y_pred.append(predicted_rating)

# Compute RMSE
rmse = mean_squared_error(y_true, y_pred, squared=False)
print(f"RMSE on the test set: {rmse:.4f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Optional: Sample the data to reduce computation time
ratings_sampled = ratings.sample(n=100000, random_state=42)

# Split the data into training and test sets
train_data, test_data = train_test_split(ratings_sampled, test_size=0.2, random_state=42)

# Create a user-item interaction matrix for the training data
train_user_item_matrix = train_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Create a user-item interaction matrix for the test data (for evaluation purposes)
test_user_item_matrix = test_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Ensure that the userIds and movieIds in the test set are present in the training set
test_data = test_data[test_data['userId'].isin(train_user_item_matrix.index) &
                      test_data['movieId'].isin(train_user_item_matrix.columns)]

# Re-create the test_user_item_matrix after filtering
test_user_item_matrix = test_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Function to compute cosine similarity between users
def cosine_similarity(matrix):
    # Normalize the matrix (subtract mean user ratings)
    norm = np.linalg.norm(matrix, axis=1)
    # Avoid division by zero
    norm[norm == 0] = 1e-10
    similarity_matrix = np.dot(matrix, matrix.T) / (norm[:, None] * norm[None, :])
    # Replace NaN values with 0
    similarity_matrix = np.nan_to_num(similarity_matrix)
    np.fill_diagonal(similarity_matrix, 0)
    return similarity_matrix

# Function to compute Pearson similarity between users
def pearson_similarity(matrix):
    mean_user_rating = np.mean(matrix, axis=1, keepdims=True)
    mean_centered = matrix - mean_user_rating
    # Compute the numerator and denominator
    numerator = np.dot(mean_centered, mean_centered.T)
    denominator = np.linalg.norm(mean_centered, axis=1)[:, None] * np.linalg.norm(mean_centered, axis=1)[None, :]
    # Avoid division by zero
    denominator[denominator == 0] = 1e-10
    similarity_matrix = numerator / denominator
    # Replace NaN values with 0
    similarity_matrix = np.nan_to_num(similarity_matrix)
    np.fill_diagonal(similarity_matrix, 0)
    return similarity_matrix

# Function to compute MSD similarity between users
def msd_similarity(matrix):
    num_users = matrix.shape[0]
    similarity_matrix = np.zeros((num_users, num_users))

    print("Computing MSD similarity matrix...")
    for i in range(num_users):
        diff = matrix[i] - matrix  # Difference between user i and all other users
        mse = np.mean((diff) ** 2, axis=1)  # Mean Squared Error
        similarity = 1 / (1 + mse)  # Convert MSE to similarity
        similarity_matrix[i] = similarity
        similarity_matrix[i, i] = 0  # Set self-similarity to 0

        # Progress indicator
        if i % 100 == 0 or i == num_users - 1:
            print(f"Processed {i + 1}/{num_users} users")

    return similarity_matrix

# Compute similarity matrices
user_similarity_cosine = cosine_similarity(train_user_item_matrix.values)
user_similarity_pearson = pearson_similarity(train_user_item_matrix.values)
user_similarity_msd = msd_similarity(train_user_item_matrix.values)

# Function to find k nearest neighbors for a user
def find_knn(user_index, k, similarity_matrix):
    # Get the similarity scores for the user
    sim_scores = similarity_matrix[user_index]
    # Find the indices of the top k similar users
    knn_indices = np.argsort(sim_scores)[-k:]
    # Reverse the array to have the most similar users first
    knn_indices = knn_indices[::-1]
    return knn_indices

# Function to predict rating for a user and item using KNN collaborative filtering
def predict_rating(user_id, item_id, k=5, similarity_matrix=user_similarity_cosine):
    if user_id not in train_user_item_matrix.index or item_id not in train_user_item_matrix.columns:
        # Return global mean if user or item is unknown
        return train_data['rating'].mean()

    user_index = train_user_item_matrix.index.get_loc(user_id)
    item_index = train_user_item_matrix.columns.get_loc(item_id)

    # Find k nearest neighbors of the user
    knn_users = find_knn(user_index, k, similarity_matrix)

    # Get the similarity scores of the k nearest neighbors
    sim_scores = similarity_matrix[user_index, knn_users]

    # Get the ratings of the k nearest neighbors for the item
    neighbor_ratings = train_user_item_matrix.values[knn_users, item_index]

    # Filter out neighbors who haven't rated the item
    mask = neighbor_ratings > 0
    sim_scores = sim_scores[mask]
    neighbor_ratings = neighbor_ratings[mask]

    if np.sum(sim_scores) > 0:
        predicted_rating = np.dot(sim_scores, neighbor_ratings) / np.sum(sim_scores)
    else:
        # If no neighbors have rated the item, return the user's average rating
        user_ratings = train_user_item_matrix.loc[user_id].replace(0, np.NaN)
        predicted_rating = user_ratings.mean()
        if np.isnan(predicted_rating):
            # If the user hasn't rated any items, return the global mean
            predicted_rating = train_data['rating'].mean()

    return predicted_rating

# Function to count users who have rated items and those who haven't
def count_users(matrix):
    # Users who have rated at least one item
    users_done = np.sum(np.sum(matrix > 0, axis=1) > 0)
    # Users who haven't rated any items
    users_not_done = matrix.shape[0] - users_done
    return users_done, users_not_done

# Count the number of users
users_done, users_not_done = count_users(train_user_item_matrix.values)
print(f"Number of users who have rated items: {users_done}")
print(f"Number of users who haven't rated any items: {users_not_done}")

# Evaluate the model on the test set for each similarity measure
methods = {
    'Cosine': user_similarity_cosine,
    'Pearson': user_similarity_pearson,
    'MSD': user_similarity_msd
}

for method_name, similarity_matrix in methods.items():
    y_true = []
    y_pred = []
    print(f"\nEvaluating using {method_name} similarity...")

    for index, row in test_data.iterrows():
        user_id = row['userId']
        item_id = row['movieId']
        true_rating = row['rating']
        predicted_rating = predict_rating(user_id, item_id, k=5, similarity_matrix=similarity_matrix)
        y_true.append(true_rating)
        y_pred.append(predicted_rating)

    rmse = mean_squared_error(y_true, y_pred, squared=False)
    print(f"RMSE on the test set using {method_name} similarity: {rmse:.4f}")
